# Homework 3 – Functions I


In [2]:
import pandas as pd
import numpy as np
from scipy import stats

flights = pd.read_csv("nycflights.csv")
ged = pd.read_csv("3 ged_data.csv")
flights.head()

,Unnamed: 0,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,1,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00
1,2,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00
2,3,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00
3,4,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00
4,5,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00


## Q1. Function returning minimum, median, maximum

In [3]:
def min_median_max(x):
    return pd.Series({"min": x.min(), "median": x.median(), "max": x.max()})

q1 = min_median_max(flights["month"])
q1

min        1.0
median     7.0
max       12.0
dtype: float64

## Q2. Explain your reasoning for choosing your function's name in the previous question.

The name clearly states the three values returned in the function, the min, median and max of the vector.

## Q3.  Write a function that categorizes a numerical variable in the flights data into four categories.

In [4]:
def categorize_time(df, col):
    s = df[col]
    conditions = [
        (s >= 500) & (s < 1200),
        (s >= 1200) & (s < 1700),
        (s >= 1700) & (s < 2100)
    ]
    choices = ["Morning", "Afternoon", "Evening"]
    return pd.Series(np.select(conditions, choices, default="Night"), index=df.index)

flights["dep_category"] = categorize_time(flights, "dep_time")
freq = flights["dep_category"].value_counts().to_frame("count")
freq.head()
freq.tail()

,count
dep_category,
Morning,129539
Afternoon,98617
Evening,79793
Night,28827


## Q4. Explain your reasoning for choosing your function's name in the previous question.

I use the name categorize_time because the purpose of the function is to categorize different times of the function based on different time frame that they are in.

## Q5. Write a function that calculates the median of all numeric variables in the flights dataset.

In [5]:
def numeric_medians(df):
    return df.select_dtypes(include=np.number).median().to_frame("median")

medians = numeric_medians(flights)
medians.head()
medians.tail()

,median
flight,1496.0
air_time,129.0
distance,872.0
hour,13.0
minute,29.0


## Q6.Explain your reasoning for choosing your function's name in the previous question.

I use the name numeric_medians because the purpose of the function is to calculate the medians of a dataframe and return a numetric output.

## Q7. Modify the function t_test() we wrote in class together this week so that this function can handle violations to the homogeneity of variance (HOV) assumption. This assumption is described below in case you are not familiar with it. Please read the class activity carefully - as all of those details are relevant.

In [6]:
def t_test(data, num_var, bin_var):
    groups = sorted(data[bin_var].dropna().unique())
    g1 = data.loc[data[bin_var] == groups[0], num_var].dropna()
    g2 = data.loc[data[bin_var] == groups[1], num_var].dropna()

    mean1, mean2 = g1.mean(), g2.mean()
    var1, var2 = g1.var(ddof=1), g2.var(ddof=1)
    n1, n2 = len(g1), len(g2)
    ratio = var1 / var2 if var2 != 0 else np.inf

    if 0.25 <= ratio <= 4:
        print("The homogeneity of variance assumption was not violated, so an independent samples t-test was conducted")
        t_stat, p_val = stats.ttest_ind(g1, g2, equal_var=True)
        dfree = n1 + n2 - 2
        test_name = "Independent samples t-test"
    else:
        print("The homogeneity of variance assumption was violated, so Welch's t-test was conducted")
        t_stat, p_val = stats.ttest_ind(g1, g2, equal_var=False)
        dfree = ((var1/n1 + var2/n2)**2) / ((((var1/n1)**2)/(n1-1)) + (((var2/n2)**2)/(n2-1)))
        test_name = "Welch's t-test"

    return pd.DataFrame({
        "mean_group1":[mean1],
        "mean_group2":[mean2],
        "var_group1":[var1],
        "var_group2":[var2],
        "n_group1":[n1],
        "n_group2":[n2],
        "t_stat":[t_stat],
        "df":[dfree],
        "test":[test_name]
    })

## Q8.  Import the GED data set we used for the class activity. Call the t_test() function and test it out on the GED data set we used in class. Let the numeric variable be 'income_log' and let the binary variable be 'ged'.

In [7]:
ged_result = t_test(ged, "income_log", "ged")
ged_result.head()
ged_result.tail()

The homogeneity of variance assumption was not violated, so an independent samples t-test was conducted


,mean_group1,mean_group2,var_group1,var_group2,n_group1,n_group2,t_stat,df,test
0,10.082916,9.606616,0.878,1.213657,5749,227,7.457946,5974,Independent samples t-test
